<a href="https://colab.research.google.com/github/joaocanaslopes/AV_project-/blob/main/analise_impacto_climatico_agricola.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análise de Impacto Climático na Produtividade Agrícola
## Estudo de Caso: Alentejo e Algarve (2005 - 2018)

Este notebook contém uma análise completa e estruturada do impacto das condições climáticas — especificamente a seca medida através do índice SPI-12 — na produtividade de diversas culturas agrícolas nas regiões do Alentejo e do Algarve.

---

### 1. Preparação de Dados e Engenharia de Funcionalidades
Nesta célula, carregamos o conjunto de dados, classificamos as intensidades de seca com base no SPI-12, calculamos um índice de produtividade normalizado (permitindo comparar culturas com escalas de produção diferentes) e criamos uma variável de atraso (*lag*) para estudar os efeitos cumulativos do clima do ano anterior.

In [ ]:
import pandas as pd
import numpy as np

# 1. Carregar os dados
df = pd.read_csv('total_com_spi12.csv')
print(f"Dataset carregado com sucesso! Linhas: {df.shape[0]}, Colunas: {df.shape[1]}")

# 2. Criar a classificação do SPI-12 (Classes de Seca)
def classify_spi(spi):
    if spi <= -2:
        return "Seca extrema"
    elif spi <= -1.5:
        return "Seca severa"
    elif spi <= -1:
        return "Seca moderada"
    elif spi >= 1:
        return "Ano húmido"
    else:
        return "Normal"

df['drought_class'] = df['spi_12'].apply(classify_spi)

# 3. Normalizar a produtividade por cultura e região (Produtividade Relativa)
mean_prod = df.groupby(['region', 'species'])['productivity_kg_ha'].transform('mean')
df['productivity_index'] = np.where(mean_prod > 0, df['productivity_kg_ha'] / mean_prod, 0)

# 4. Criar o Efeito Retardado (Lag Effect) do SPI-12 do ano anterior
df = df.sort_values(by=['region', 'species', 'ano'])
df['spi_12_lag1'] = df.groupby(['region', 'species'])['spi_12'].shift(1)

# Visualizar sumário dos dados
df.head()

### 2. Análise Exploratória e Visualizações
Vamos analisar visualmente o comportamento da produtividade agrícola face às diferentes classes de seca e mapear a evolução temporal através de um Heatmap focado em culturas-chave.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]

# Culturas-chave recomendadas para análise focada
culturas_chave = ['Trigo', 'Milho', 'Aveia', 'Cevada', 'Olival', 'Vinha', 'Batata']
df_filtrado = df[df['species'].isin(culturas_chave)]

# --- Visualização 1: Boxplot da Produtividade por Classe de Seca ---
plt.figure()
ordem_classes = ["Seca extrema", "Seca severa", "Seca moderada", "Normal", "Ano húmido"]
sns.boxplot(
    data=df_filtrado,
    x='drought_class',
    y='productivity_index',
    hue='region',
    order=ordem_classes
)
plt.title('Impacto das Classes de Seca no Índice de Produtividade (Culturas Selecionadas)')
plt.ylabel('Índice de Produtividade (1.0 = Média Histórica)')
plt.xlabel('Condição Climática (SPI-12)')
plt.axhline(1.0, color='red', linestyle='--', alpha=0.7)
plt.show()

# --- Visualização 2: Heatmap de Vulnerabilidade das Culturas (Alentejo) ---
df_alentejo = df_filtrado[df_filtrado['region'] == 'Alentejo']
pivot_prod = df_alentejo.pivot_table(index='species', columns='ano', values='productivity_index')

plt.figure(figsize=(14, 6))
sns.heatmap(pivot_prod, annot=True, fmt=".2f", cmap="RdYlGn", center=1.0)
title_font = {'size': 14, 'weight': 'bold'}
plt.title('Evolução do Índice de Produtividade por Cultura no Alentejo (Vermelho = Quebra)', fontdict=title_font)
plt.xlabel('Ano')
plt.ylabel('Cultura')
plt.show()

### 3. Modelagem Estatística Avançada e Efeito "Tampão" da Irrigação
Aqui aplicamos uma regressão linear via OLS (Ordinary Least Squares) para testar se a área irrigável atua como um mitigador (efeito tampão) dos impactos da seca através de um termo de interação (`spi_12:irrigable_area_k_ha`). Incluímos também o efeito de atraso do clima (`spi_12_lag1`).

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Remover valores nulos gerados pela coluna de Lag
df_model = df_filtrado.dropna(subset=['spi_12_lag1', 'productivity_index']).copy()

# Normalizar a área irrigável por mil para melhor escala dos coeficientes
df_model['irrigable_area_k_ha'] = df_model['irrigable_area_ha'] / 1000

print("A executar o modelo de regressão linear...")
modelo = smf.ols(
    formula="productivity_index ~ spi_12 + spi_12_lag1 + mean_temp + irrigable_area_k_ha + spi_12:irrigable_area_k_ha",
    data=df_model
).fit()

print(modelo.summary())

### 4. Análise de Importância das Variáveis (Random Forest)
Utilizamos um modelo de Machine Learning baseado em árvores para mensurar de forma puramente matemática qual o peso de cada variável explicativa nas variações do índice de produtividade.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

features = ['spi_12', 'spi_12_lag1', 'mean_temp', 'precipitation_mm', 'irrigable_area_ha']
df_rf = df.dropna(subset=['productivity_index'] + features).copy()

X = df_rf[features]
y = df_rf['productivity_index']

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X, y)

df_importances = pd.DataFrame({'Variável': X.columns, 'Importância': rf_model.feature_importances_})
df_importances = df_importances.sort_values(by='Importância', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=df_importances, x='Importância', y='Variável', palette='viridis')
plt.title('Importância das Variáveis na Produtividade Agrícola (Random Forest)')
plt.xlabel('Peso de Importância (0 a 1)')
plt.ylabel('Variável')
plt.show()

### 5. Agrupamento Automático de Culturas (K-Means Clustering)
Para classificar as 94 culturas de forma analítica, aplicamos o algoritmo K-Means para criar três agrupamentos baseados no risco/volatilidade da produtividade e na dependência histórica de regadio.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

df_alentejo_all = df[df['region'] == 'Alentejo']
perfil_culturas = df_alentejo_all.groupby('species').agg(
    prod_media_index=('productivity_index', 'mean'),
    prod_volatilidade=('productivity_index', 'std'),
    area_rega_media=('irrigable_area_ha', 'mean')
).dropna()

scaler = StandardScaler()
perfil_scaled = scaler.fit_transform(perfil_culturas)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
perfil_culturas['Grupo_KMeans'] = kmeans.fit_predict(perfil_scaled)

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=perfil_culturas,
    x='prod_volatilidade',
    y='area_rega_media',
    hue='Grupo_KMeans',
    palette='Set1',
    s=100
)

for i in range(0, len(perfil_culturas), 5):
    plt.text(
        perfil_culturas['prod_volatilidade'].iloc[i] + 0.01,
        perfil_culturas['area_rega_media'].iloc[i],
        perfil_culturas.index[i],
        fontsize=9
)

plt.title('Clustering de Culturas: Volatilidade da Produtividade vs Dependência de Rega')
plt.xlabel('Volatilidade da Produtividade (Risco)')
plt.ylabel('Área de Rega Média (Hectares)')
plt.legend(title='Grupo (Cluster)')
plt.show()

### 6. Estimativa de Impacto Económico
Por fim, traduzimos a perda física (toneladas que deixaram de ser produzidas face à média esperada) em valores monetários estimados para as culturas principais durante a grande seca de 2005 no Alentejo.

In [ ]:
precos_por_tonelada = {
    'Trigo': 250,
    'Milho': 280,
    'Tomate para indústria': 85,
    'Batata': 350,
    'Vinha': 800
}

df_eco = df[df['species'].isin(precos_por_tonelada.keys())].copy()
df_eco['preco_t_estimado'] = df_eco['species'].map(precos_por_tonelada)

# Produção esperada com base na média histórica regional
df_eco['producao_esperada_t'] = df_eco['area_ha'] * (df_eco['productivity_kg_ha'] / df_eco['productivity_index']) / 1000
df_eco['quebra_producao_t'] = (df_eco['producao_esperada_t'] - df_eco['production_t']).clip(lower=0)
df_eco['receita_perdida_euros'] = df_eco['quebra_producao_t'] * df_eco['preco_t_estimado']

seca_2005 = df_eco[(df_eco['ano'] == 2005) & (df_eco['region'] == 'Alentejo')]
impacto_financeiro = seca_2005.groupby('species')['receita_perdida_euros'].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
impacto_financeiro.plot(kind='bar', color='firebrick')
plt.title('Estimativa de Receita Perdida por Cultura no Alentejo - Seca de 2005')
plt.ylabel('Euros Perdidos')
plt.xlabel('Cultura')
plt.xticks(rotation=45)
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, loc: "{:,} M€".format(x/1000000)))
plt.show()